## NB10 — Validação com CEIS e Análise Qualitativa das Propostas de Alto Risco

**Entradas:**
- `data/output/ranking_parcial.csv` — ranking de risco gerado pelo NB09 (75.471 propostas)
- `data/processed/dataset_ml.csv` — data de publicação (via `dias_desde_eleicao`) e tipo de fornecedor de cada proposta (NB05)
- `data/raw/dados_CEIS.csv` — Cadastro de Empresas Inidôneas e Suspensas (CGU, Set/2026)

**Objetivo:**
- Cruzar o ranking de risco com o CEIS para obter um ground truth parcial
- Separar a sanção **vigente na data da proposta** da sanção **posterior** à proposta (as propostas são de 2023–2024 e o CEIS é de 2026)
- Casar por CNPJ completo e por CNPJ raiz
- Repetir a comparação por **fornecedor** (teste exato de Fisher), além da comparação por proposta


---
### 0. Imports e Configuração

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, hypergeom

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Setup OK")

Setup OK


---
### 1. Carregamento dos Dados

#### 1.1 Ranking Parcial (NB09)

In [2]:
ranking = pd.read_csv('data/output/ranking_parcial.csv')

ranking['cnpj_str'] = ranking['cnpj_forn_norm'].astype(str).str.zfill(14)

print(f'Shape: {ranking.shape}')
print(f'Colunas: {ranking.columns.tolist()}')
print()
print(ranking[['posicao_risco', 'cnpj_str', 'votos', 'score_combinado']].head(5))

Shape: (75471, 19)
Colunas: ['orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'cnpj_forn_norm', 'if_score', 'if_label', 'if_score_norm', 'lof_score', 'lof_label', 'lof_score_norm', 'lof_n_copias', 'ocsvm_score', 'ocsvm_label', 'ocsvm_score_norm', 'votos', 'score_combinado', 'posicao_risco', 'cnpj_str']

   posicao_risco        cnpj_str  votos  score_combinado
0              1  52105018000151      2           0.5889
1              2  52934421000193      2           0.5825
2              3  07199135000177      2           0.5625
3              4  51026267000199      2           0.5591
4              5  51026267000199      2           0.5571


#### 1.3 Data da proposta e tipo de fornecedor

O ranking não traz a data. A data de referência é a de **publicação da contratação no PNCP** (NB05, seção 5.3), recuperada de `dias_desde_eleicao` (dias até 06/10/2024, 1º turno) em `data/processed/dataset_ml.csv`.
A data é única por contratação (órgão, ano e sequencial); a chave completa da proposta (com item e fornecedor) se repete em algumas linhas, por isso a junção é feita pela contratação.

O tipo de fornecedor (`pessoa_fisica` ou `pessoa_juridica`) vem do mesmo arquivo. A **raiz do CNPJ** (8 primeiros dígitos) só é definida para pessoa jurídica: um CPF preenchido com zeros até 14 dígitos não tem raiz.


In [3]:
JK = ['orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp']
ELEICAO = pd.Timestamp(2024, 10, 6)       # mesma referência do NB05: dias_desde_eleicao = data de publicação - 06/10/2024

ml = pd.read_csv('data/processed/dataset_ml.csv', usecols=JK + ['cnpj_forn_norm', 'dias_desde_eleicao', 'tipo_fornecedor'])

# uma data por contratação
assert (ml.groupby(JK)['dias_desde_eleicao'].nunique() == 1).all(), 'a data deveria ser única por contratação'
datas = ml.drop_duplicates(JK)[JK + ['dias_desde_eleicao']]
n_antes = len(ranking)
ranking = ranking.merge(datas, on=JK, how='left', validate='many_to_one')
assert len(ranking) == n_antes
ranking['data_proposta'] = ELEICAO + pd.to_timedelta(ranking['dias_desde_eleicao'], unit='D')

# um tipo por CNPJ/CPF
assert (ml.groupby('cnpj_forn_norm')['tipo_fornecedor'].nunique() == 1).all()
tipo = ml.drop_duplicates('cnpj_forn_norm').set_index('cnpj_forn_norm')['tipo_fornecedor']
assert ranking['cnpj_forn_norm'].map(tipo).notna().all() and ranking['data_proposta'].notna().all()
ranking['pj'] = ranking['cnpj_forn_norm'].map(tipo).eq('pessoa_juridica')
ranking['raiz'] = ranking['cnpj_str'].str[:8].where(ranking['pj'])      # raiz só para pessoa jurídica

print(f'Data da proposta: {ranking["data_proposta"].min():%d/%m/%Y} a {ranking["data_proposta"].max():%d/%m/%Y}')
print(f'Propostas de pessoa jurídica: {ranking["pj"].sum():,} ({ranking["pj"].mean():.1%}) | pessoa física: {(~ranking["pj"]).sum():,} ({(~ranking["pj"]).mean():.1%})')
print(f'Fornecedores distintos: {ranking["cnpj_str"].nunique():,} (CNPJ/CPF) | PJ por raiz: {ranking["raiz"].nunique():,}')


Data da proposta: 04/01/2023 a 30/12/2024
Propostas de pessoa jurídica: 68,699 (91.0%) | pessoa física: 6,772 (9.0%)
Fornecedores distintos: 9,437 (CNPJ/CPF) | PJ por raiz: 6,672


#### 1.2 CEIS — Cadastro de Empresas Inidôneas e Suspensas

In [4]:
ceis = pd.read_csv('data/raw/dados_CEIS.csv', sep=';', encoding='latin1')

print(f'Shape: {ceis.shape}')
print(f'Tipo de pessoa: {ceis["TIPO DE PESSOA"].value_counts().to_dict()}')

Shape: (23706, 24)
Tipo de pessoa: {'J': 14533, 'F': 9161}


---
### 2. Limpeza dos Dados

#### 2.1 Filtro: Apenas Pessoas Jurídicas + Limpeza de CNPJs

In [5]:
ceis_pj = ceis[ceis['TIPO DE PESSOA'] == 'J'].copy()

ceis_pj['cnpj_clean'] = (
    ceis_pj['CPF OU CNPJ DO SANCIONADO']
    .astype(str)
    .str.replace(r'\D', '', regex=True)
    .str.zfill(14)
)

ceis_pj['raiz'] = ceis_pj['cnpj_clean'].str[:8]      # raiz do CNPJ (8 primeiros dígitos)

print(f'PJ total: {len(ceis_pj):,}  |  CNPJs únicos: {ceis_pj["cnpj_clean"].nunique():,}')
print(f'Raízes de CNPJ distintas: {ceis_pj["raiz"].nunique():,}')


PJ total: 14,533  |  CNPJs únicos: 9,606
Raízes de CNPJ distintas: 9,571


#### 2.2 Datas das sanções

Converte `DATA INÍCIO SANÇÃO` e `DATA FINAL SANÇÃO` e monta dois conjuntos:
- `cnpjs_ceis_todos`: todos os CNPJs sancionados historicamente (2003–2026);
- `cnpjs_ceis_ativos`: **classificação anterior**: CNPJs com sanção ativa em 05/10/2026 (data final ausente ou posterior a essa data). Ela é mantida na seção 3.1 só para comparação: 05/10/2026 é posterior a todas as propostas (2023–2024).
  A análise pela data de cada proposta está na seção 3.2.

Sanções **sem data final** (sem prazo determinado, ou com prazo não informado) são tratadas como sem término. A data de início é obrigatória em todas as sanções.


In [6]:
ceis_pj['DATA INÍCIO SANÇÃO'] = pd.to_datetime(
    ceis_pj['DATA INÍCIO SANÇÃO'], format='%d/%m/%Y', errors='coerce'
)
ceis_pj['DATA FINAL SANÇÃO'] = pd.to_datetime(
    ceis_pj['DATA FINAL SANÇÃO'], format='%d/%m/%Y', errors='coerce'
)

DATA_REFERENCIA = pd.Timestamp('2026-10-05')

ceis_ativos = ceis_pj[
    ceis_pj['DATA FINAL SANÇÃO'].isna() |
    (ceis_pj['DATA FINAL SANÇÃO'] >= DATA_REFERENCIA)
]

cnpjs_ceis_todos  = set(ceis_pj['cnpj_clean'].unique())
cnpjs_ceis_ativos = set(ceis_ativos['cnpj_clean'].unique())

print(f'CNPJs CEIS — histórico completo:  {len(cnpjs_ceis_todos):,}')
print(f'CNPJs CEIS — sanção ativa:        {len(cnpjs_ceis_ativos):,}')

n_sem_fim = ceis_pj['DATA FINAL SANÇÃO'].isna().sum()
n_depois = (ceis_pj['DATA INÍCIO SANÇÃO'] > pd.Timestamp('2024-12-30')).sum()
print(f'Início das sanções: {ceis_pj["DATA INÍCIO SANÇÃO"].min():%d/%m/%Y} a {ceis_pj["DATA INÍCIO SANÇÃO"].max():%d/%m/%Y}')
print(f'Sanções sem data final: {n_sem_fim:,} ({n_sem_fim / len(ceis_pj):.1%})')
print(f'Sanções iniciadas depois da última proposta (30/12/2024): {n_depois:,} ({n_depois / len(ceis_pj):.1%})')


CNPJs CEIS — histórico completo:  9,606
CNPJs CEIS — sanção ativa:        9,108
Início das sanções: 28/01/2003 a 28/09/2026
Sanções sem data final: 1,790 (12.3%)
Sanções iniciadas depois da última proposta (30/12/2024): 9,926 (68.3%)


---
### 3. Cruzamento Ranking × CEIS

#### 3.1 Classificação anterior: sanção ativa em 05/10/2026

Cruza pelo CNPJ completo com a sanção "ativa" em 05/10/2026, sem considerar a data da proposta. Mantida para comparação com a seção 3.2.


In [7]:
ranking['no_ceis_total'] = ranking['cnpj_str'].isin(cnpjs_ceis_todos).astype(int)
ranking['no_ceis_ativo']  = ranking['cnpj_str'].isin(cnpjs_ceis_ativos).astype(int)

print('=== Resultado do Cruzamento ===')
print()
print(f'Total de propostas:                 {len(ranking):,}')
print(f'CNPJs únicos no dataset:            {ranking["cnpj_str"].nunique():,}')
print(f'CNPJs únicos no CEIS (histórico):   {len(cnpjs_ceis_todos):,}')
print()
n_tot = ranking['no_ceis_total'].sum()
n_ati = ranking['no_ceis_ativo'].sum()
print(f'Propostas c/ fornecedor no CEIS (histórico): {n_tot:,}  ({n_tot/len(ranking)*100:.2f}%)')
print(f'Propostas c/ fornecedor no CEIS (ativo):     {n_ati:,}  ({n_ati/len(ranking)*100:.2f}%)')
print()
print('--- Por número de votos ---')
print(ranking.groupby('votos')[['no_ceis_total', 'no_ceis_ativo']].agg(['sum', 'mean']).round(4))

=== Resultado do Cruzamento ===

Total de propostas:                 75,471
CNPJs únicos no dataset:            9,437
CNPJs únicos no CEIS (histórico):   9,606

Propostas c/ fornecedor no CEIS (histórico): 11,674  (15.47%)
Propostas c/ fornecedor no CEIS (ativo):     11,308  (14.98%)

--- Por número de votos ---
      no_ceis_total        no_ceis_ativo       
                sum   mean           sum   mean
votos                                          
0             11454 0.1559         11089 0.1509
1               214 0.1257           213 0.1251
2                 6 0.0196             6 0.0196


In [8]:
alto_risco = ranking[ranking['votos'] >= 2].copy()

print(f'Propostas de alto risco (votos ≥ 2): {len(alto_risco):,}')
print(f'  → CEIS ativo: {alto_risco["no_ceis_ativo"].sum()} ({alto_risco["no_ceis_ativo"].mean()*100:.1f}%)')
print()

baseline_rate = ranking['no_ceis_ativo'].mean()
expected_random = baseline_rate * len(alto_risco)
print(f'Taxa de CEIS na base completa (baseline): {baseline_rate*100:.1f}%')
print(f'Hits esperados por seleção aleatória:     {expected_random:.0f}')
print(f'Hits observados:                          {alto_risco["no_ceis_ativo"].sum()}')
print()

for v in [3, 2]:
    sub = ranking[ranking['votos'] == v]
    print(f'votos={v} ({len(sub)} propostas): {sub["no_ceis_ativo"].sum()} no CEIS ativo ({sub["no_ceis_ativo"].mean()*100:.1f}%)')

Propostas de alto risco (votos ≥ 2): 306
  → CEIS ativo: 6 (2.0%)

Taxa de CEIS na base completa (baseline): 15.0%
Hits esperados por seleção aleatória:     46
Hits observados:                          6

votos=3 (0 propostas): 0 no CEIS ativo (nan%)
votos=2 (306 propostas): 6 no CEIS ativo (2.0%)


#### 3.2 Sanção vigente na data da proposta × sanção posterior

As propostas são de 2023 e 2024, e a classificação da seção 3.1 foi feita em 05/10/2026: uma sanção iniciada em 2025 conta como "ativa", mas não existia na época da proposta. Aqui cada proposta é comparada com as datas das sanções do seu fornecedor:

| Situação | Regra | Interpretação |
|---|---|---|
| **Vigente na data** | início ≤ data da proposta e (sem data final ou final ≥ data da proposta) | empresa impedida de contratar que venceu: sinal forte |
| **Posterior** | início > data da proposta | o fornecedor foi sancionado depois: teste de detecção antecipada |
| **Posterior em até 12 meses** | posterior, com início até 365 dias depois da proposta | janela fixa: o CEIS vai até 05/10/2026 e as propostas de dezembro de 2024 têm só 21 meses de exposição, contra mais de 40 para as de 2023 |
| **Encerrada antes** | data final < data da proposta | sanção que já havia expirado |

Um fornecedor pode ter sanções de mais de um tipo: as marcas acima são independentes. A **classe exclusiva** segue a ordem vigente > posterior > encerrada antes > sem registro.

**Chave de correspondência.** CNPJ **raiz** (principal): a sanção atinge a empresa, e as filiais têm outro CNPJ completo. O CNPJ completo é mostrado para comparação. A raiz só vale para pessoa jurídica.


In [9]:
def situacao_sancao(chave):
    """Situação das sanções do fornecedor de cada proposta em relação à data da proposta.
    `chave`: 'raiz' (8 primeiros dígitos, só PJ) ou 'cnpj_str' (CNPJ completo)."""
    ce = 'raiz' if chave == 'raiz' else 'cnpj_clean'
    san = ceis_pj[[ce, 'DATA INÍCIO SANÇÃO', 'DATA FINAL SANÇÃO', 'CATEGORIA DA SANÇÃO', 'ABRAGÊNCIA DA SANÇÃO', 'ESFERA ÓRGÃO SANCIONADOR']].rename(
        columns={ce: 'chave', 'DATA INÍCIO SANÇÃO': 'inicio', 'DATA FINAL SANÇÃO': 'fim'})
    m = (ranking[['data_proposta', chave]].rename(columns={chave: 'chave'}).reset_index().dropna(subset=['chave'])
         .merge(san, on='chave', how='inner'))
    m['atraso_dias'] = (m['inicio'] - m['data_proposta']).dt.days
    m['vigente'] = (m['inicio'] <= m['data_proposta']) & (m['fim'].isna() | (m['fim'] >= m['data_proposta']))
    m['posterior'] = m['inicio'] > m['data_proposta']
    m['posterior_12m'] = m['posterior'] & (m['atraso_dias'] <= 365)
    m['encerrada_antes'] = m['fim'].notna() & (m['fim'] < m['data_proposta'])
    marcas = m.groupby('index')[['vigente', 'posterior', 'posterior_12m', 'encerrada_antes']].any()
    out = pd.DataFrame(False, index=ranking.index, columns=['qualquer'] + list(marcas.columns))
    out.loc[marcas.index, marcas.columns] = marcas
    out.loc[marcas.index, 'qualquer'] = True
    out['classe'] = np.select([out['vigente'], out['posterior'], out['encerrada_antes']],
                              ['Vigente na data', 'Só posterior', 'Só encerrada antes'], 'Sem registro')
    out['atraso_meses'] = (m[m['posterior']].groupby('index')['atraso_dias'].min() / 30.44).reindex(out.index)
    return out, m


situacao, casamentos = {}, {}
for nome, chave in [('CNPJ raiz', 'raiz'), ('CNPJ completo', 'cnpj_str')]:
    situacao[nome], casamentos[nome] = situacao_sancao(chave)

alto = (ranking['votos'] >= 2).to_numpy()
ORDEM = ['Vigente na data', 'Só posterior', 'Só encerrada antes', 'Sem registro']
MARCAS = [('Qualquer sanção (histórico)', 'qualquer'), ('Vigente na data da proposta', 'vigente'), ('Posterior à proposta', 'posterior'),
          ('Posterior em até 12 meses', 'posterior_12m'), ('Encerrada antes da proposta', 'encerrada_antes')]

saida = []
for nome, s in situacao.items():
    t = pd.DataFrame({'Todas as propostas': s['classe'].value_counts(), f'Alto risco (>= 2 votos, n={alto.sum()})': s.loc[alto, 'classe'].value_counts()}).reindex(ORDEM).fillna(0).astype(int)
    t['% da base'] = (t.iloc[:, 0] / len(ranking) * 100).map('{:.2f}'.format)
    t['% do alto risco'] = (t.iloc[:, 1] / alto.sum() * 100).map('{:.2f}'.format)
    print(f'=== Classe exclusiva da proposta — {nome} ===')
    print(t.to_string()); print()
    for classe, row in t.iterrows():
        saida.append({'chave': nome, 'classe': classe, 'todas': row.iloc[0], 'alto_risco': row.iloc[1]})

print('=== Marcas independentes (uma proposta pode ter mais de uma) ===')
marc = pd.DataFrame({(nome, g): [int(s[c].sum()) if g == 'Todas' else int(s.loc[alto, c].sum()) for _, c in MARCAS]
                     for nome, s in situacao.items() for g in ('Todas', 'Alto risco')}, index=[r for r, _ in MARCAS])
print(marc.to_string())
pd.DataFrame(saida).to_csv('data/output/ceis_situacao_sancao.csv', index=False)

print('\n=== Classificação anterior (ativa em 05/10/2026) × situação na data da proposta (CNPJ completo, mesma chave de 3.1) ===')
ativa_ant = ranking['no_ceis_ativo'].map({1: 'Ativa em 05/10/2026', 0: 'Não ativa em 05/10/2026'})
print(pd.crosstab(ativa_ant, situacao['CNPJ completo']['classe']).reindex(columns=ORDEM).fillna(0).astype(int).to_string())

print('\n=== CNPJ raiz × CNPJ completo: propostas a mais com a raiz ===')
for r, c in MARCAS[:4]:
    a, b = int(situacao['CNPJ completo'][c].sum()), int(situacao['CNPJ raiz'][c].sum())
    print(f'{r:30s} completo {a:6,} | raiz {b:6,} | a mais: {b - a:4,}')
print(f'Raízes de CNPJ com mais de um CNPJ completo sancionado: {int((ceis_pj.groupby("raiz")["cnpj_clean"].nunique() > 1).sum())}')

pos = situacao['CNPJ raiz']['atraso_meses'].dropna()
print(f'\nAtraso até a primeira sanção posterior (CNPJ raiz): mediana {pos.median():.1f} meses (quartis {pos.quantile(.25):.1f} a {pos.quantile(.75):.1f}) | {len(pos):,} propostas')

vig = casamentos['CNPJ raiz'].query('vigente').drop_duplicates('index')
print(f'\n=== As {len(vig)} propostas com sanção vigente na data (CNPJ raiz): categoria, abrangência e esfera da primeira sanção ===')
for col in ['CATEGORIA DA SANÇÃO', 'ABRAGÊNCIA DA SANÇÃO', 'ESFERA ÓRGÃO SANCIONADOR']:
    print(vig[col].value_counts(dropna=False).to_string(), '\n')


=== Classe exclusiva da proposta — CNPJ raiz ===
                    Todas as propostas  Alto risco (>= 2 votos, n=306) % da base % do alto risco
classe                                                                                          
Vigente na data                    154                               0      0.20            0.00
Só posterior                     11602                               6     15.37            1.96
Só encerrada antes                   8                               0      0.01            0.00
Sem registro                     63707                             300     84.41           98.04

=== Classe exclusiva da proposta — CNPJ completo ===
                    Todas as propostas  Alto risco (>= 2 votos, n=306) % da base % do alto risco
classe                                                                                          
Vigente na data                    144                               0      0.19            0.00
Só posterior            


=== Classificação anterior (ativa em 05/10/2026) × situação na data da proposta (CNPJ completo, mesma chave de 3.1) ===
classe                   Vigente na data  Só posterior  Só encerrada antes  Sem registro
no_ceis_ativo                                                                           
Ativa em 05/10/2026                  142         11166                   0             0
Não ativa em 05/10/2026                2           356                   8         63797

=== CNPJ raiz × CNPJ completo: propostas a mais com a raiz ===
Qualquer sanção (histórico)    completo 11,674 | raiz 11,764 | a mais:   90
Vigente na data da proposta    completo    144 | raiz    154 | a mais:   10
Posterior à proposta           completo 11,604 | raiz 11,684 | a mais:   80
Posterior em até 12 meses      completo  2,672 | raiz  2,720 | a mais:   48
Raízes de CNPJ com mais de um CNPJ completo sancionado: 28

Atraso até a primeira sanção posterior (CNPJ raiz): mediana 18.6 meses (quartis 12.7 a 23.3) | 

#### 3.3 Comparação por proposta: alto risco × base

Teste exato de Fisher (bilateral) e p unilateral "menos" = P(X ≤ k) da distribuição hipergeométrica: probabilidade de o alto risco ter **tão poucas** propostas com o desfecho quanto o observado, se fosse uma seleção ao acaso. `Odds ratio` < 1: o alto risco é *menos* sancionado que a base.

**Unidade de análise: a proposta.** Cada linha da base é uma observação; um fornecedor com muitas propostas pesa muitas vezes e as propostas do mesmo fornecedor não são independentes. A seção 4 repete a comparação por fornecedor.

**População.** *Só PJ* é a principal: o CEIS foi filtrado por pessoa jurídica, então a pessoa física nunca casa por construção e, na população, só dilui a taxa. *Todas* inclui as pessoas físicas, como na análise anterior.


In [10]:
def teste_fisher(k, N, n, M):
    """k sancionados entre os N do grupo; n entre os M da população. p unilateral = hipergeométrica P(X <= k)."""
    tab = [[k, N - k], [n - k, M - n - (N - k)]]
    orr, p_bi = fisher_exact(tab)
    return {'Grupo': f'{k}/{N}', '% grupo': 100 * k / N, 'População': f'{n}/{M}', '% população': 100 * n / M,
            'Esperado': N * n / M, 'Odds ratio': orr, 'p bilateral (Fisher)': p_bi, 'p unilateral (menos)': hypergeom.cdf(k, M, n, N)}


def fmt_tabela(df):
    d = df.copy()
    for c in ['% grupo', '% população', 'Esperado']: d[c] = d[c].map('{:.1f}'.format)
    d['Odds ratio'] = d['Odds ratio'].map('{:.3f}'.format)
    for c in ['p bilateral (Fisher)', 'p unilateral (menos)']: d[c] = d[c].map(lambda p: f'{p:.2g}')
    return d


DESFECHOS = [('Qualquer sanção (histórico)', 'qualquer'), ('Vigente na data da proposta', 'vigente'),
             ('Posterior à proposta', 'posterior'), ('Posterior em até 12 meses', 'posterior_12m')]
pj_v = ranking['pj'].to_numpy()

linhas = []
for chave, s in situacao.items():
    for pop, mask in [('Só PJ', pj_v), ('Todas', np.ones(len(ranking), dtype=bool))]:
        for rot, col in DESFECHOS:
            sel = s[col].to_numpy()
            linhas.append({'Unidade': 'Proposta', 'Chave': chave, 'População de referência': pop, 'Desfecho': rot,
                           **teste_fisher(int((sel & alto & mask).sum()), int((alto & mask).sum()), int((sel & mask).sum()), int(mask.sum()))})
teste_prop = pd.DataFrame(linhas)
teste_prop.to_csv('data/output/ceis_teste_proposta.csv', index=False)
print('=== Por proposta: alto risco (>= 2 votos) × base ===')
print(fmt_tabela(teste_prop).to_string(index=False))


=== Por proposta: alto risco (>= 2 votos) × base ===
 Unidade         Chave População de referência                    Desfecho Grupo % grupo   População % população Esperado Odds ratio p bilateral (Fisher) p unilateral (menos)
Proposta     CNPJ raiz                   Só PJ Qualquer sanção (histórico) 6/166     3.6 11764/68699        17.1     28.4      0.181              1.3e-07                7e-08
Proposta     CNPJ raiz                   Só PJ Vigente na data da proposta 0/166     0.0   154/68699         0.2      0.4      0.000                    1                 0.69
Proposta     CNPJ raiz                   Só PJ        Posterior à proposta 6/166     3.6 11684/68699        17.0     28.2      0.183              1.3e-07              8.4e-08
Proposta     CNPJ raiz                   Só PJ   Posterior em até 12 meses 2/166     1.2  2720/68699         4.0      6.6      0.295                0.071                0.038
Proposta     CNPJ raiz                   Todas Qualquer sanção (históric

---
### 4. Validação por fornecedor (teste exato de Fisher / hipergeométrico)

A comparação por proposta pode superestimar a evidência: um mesmo fornecedor pode vencer centenas de itens, e essas propostas não são independentes. Aqui a **unidade é o fornecedor distinto**:

- **Grupo:** fornecedores com **ao menos uma proposta de alto risco** (>= 2 votos).
- **População:** todos os fornecedores distintos da base.
- **Fornecedor "sancionado":** o desfecho ocorre em ao menos uma proposta sua (qualquer sanção, vigente na data, posterior, posterior em até 12 meses).
- **Fornecedor distinto:** pela **raiz do CNPJ** (principal; o CPF da pessoa física é o próprio identificador) e, para comparação, pelo CNPJ completo.
- **População de referência:** *Só PJ* (principal) e *Todos* (inclui as pessoas físicas, que nunca casam). A tabela traz as duas porque o resultado depende dessa escolha.

**Limitações:** o grupo tem poucos fornecedores, então o poder do teste é baixo; e como a seleção do alto risco é feita **por proposta**, fornecedores grandes e recorrentes têm mais chance de entrar no grupo (ver a referência ponderada por proposta ao final).


In [11]:
linhas = []
for chave_nome, col_unid in [('CNPJ raiz', 'raiz'), ('CNPJ completo', 'cnpj_str')]:
    s = situacao[chave_nome]
    unid = ranking[col_unid].where(ranking['pj'], ranking['cnpj_str'])         # PF: o próprio CPF (nunca casa)
    for pop, mask in [('Só PJ', pj_v), ('Todos', np.ones(len(ranking), dtype=bool))]:
        for rot, col in DESFECHOS:
            d = pd.DataFrame({'unid': unid[mask], 'flag': s.loc[mask, col], 'alto': alto[mask]})
            forn = d.groupby('unid').agg(flag=('flag', 'max'), alto=('alto', 'max'))
            linhas.append({'Unidade': 'Fornecedor', 'Chave': chave_nome, 'População de referência': pop, 'Desfecho': rot,
                           **teste_fisher(int(forn.loc[forn['alto'], 'flag'].sum()), int(forn['alto'].sum()), int(forn['flag'].sum()), len(forn))})
teste_forn = pd.DataFrame(linhas)
teste_forn.to_csv('data/output/ceis_teste_fornecedor.csv', index=False)
print('=== Por fornecedor: fornecedores com >= 1 proposta de alto risco × todos os fornecedores ===')
print(fmt_tabela(teste_forn).to_string(index=False))

# --- concentração (por que a unidade importa)
raiz_pj = ranking[pj_v]
prop_forn = raiz_pj.groupby('raiz').size()
sanc_raiz = situacao['CNPJ raiz'].loc[raiz_pj.index, 'qualquer'].groupby(raiz_pj['raiz']).max()
casadas = raiz_pj[situacao['CNPJ raiz'].loc[raiz_pj.index, 'qualquer']].groupby('raiz').size().sort_values(ascending=False)
alto_forn = ranking[alto].groupby(ranking['raiz'].where(ranking['pj'], ranking['cnpj_str'])).size().sort_values(ascending=False)
print('\n=== Concentração por fornecedor (CNPJ raiz, pessoa jurídica) ===')
print(f'Fornecedores PJ distintos: {len(prop_forn):,} | propostas por fornecedor: mediana {prop_forn.median():.0f}, máximo {prop_forn.max():,}')
print(f'Fornecedores com sanção no CEIS: {len(casadas):,} | propostas dessas: {casadas.sum():,} | maior fornecedor: {casadas.iloc[0]:,} propostas | '
      f'os 20 maiores respondem por {casadas.head(20).sum() / casadas.sum():.0%} das propostas com correspondência')
print(f'Propostas por fornecedor: sancionados média {prop_forn[sanc_raiz].mean():.1f} (mediana {prop_forn[sanc_raiz].median():.0f}) × demais média {prop_forn[~sanc_raiz].mean():.1f} (mediana {prop_forn[~sanc_raiz].median():.0f})')
print(f'Alto risco: {alto.sum()} propostas de {len(alto_forn)} fornecedores distintos | o maior fornecedor tem {alto_forn.iloc[0]} propostas ({alto_forn.iloc[0] / alto.sum():.0%}) | os 5 maiores, {alto_forn.head(5).sum() / alto.sum():.0%}')

# --- referência ponderada por proposta: se o alto risco fosse sorteado por proposta, que fração de fornecedores PJ seria sancionada?
rng = np.random.RandomState(1)
raiz_arr, pj_arr = ranking['raiz'].to_numpy(), ranking['pj'].to_numpy()
fr = []
for _ in range(3000):
    sorteio = rng.choice(len(ranking), int(alto.sum()), replace=False)
    sorteio = sorteio[pj_arr[sorteio]]
    fr.append(sanc_raiz.reindex(pd.unique(raiz_arr[sorteio])).mean())
print(f'\nReferência ponderada por proposta: sorteando {alto.sum()} propostas ao acaso, {np.mean(fr):.1%} dos fornecedores PJ distintos sorteados são sancionados '
      f'(IC 95% {np.percentile(fr, 2.5):.1%} a {np.percentile(fr, 97.5):.1%}); na população de fornecedores PJ, {sanc_raiz.mean():.1%}.')


=== Por fornecedor: fornecedores com >= 1 proposta de alto risco × todos os fornecedores ===
   Unidade         Chave População de referência                    Desfecho Grupo % grupo População % população Esperado Odds ratio p bilateral (Fisher) p unilateral (menos)
Fornecedor     CNPJ raiz                   Só PJ Qualquer sanção (histórico)  5/91     5.5  770/6672        11.5     10.5      0.442                 0.07                 0.04
Fornecedor     CNPJ raiz                   Só PJ Vigente na data da proposta  0/91     0.0   24/6672         0.4      0.3      0.000                    1                 0.72
Fornecedor     CNPJ raiz                   Só PJ        Posterior à proposta  5/91     5.5  763/6672        11.4     10.4      0.447                0.094                0.043
Fornecedor     CNPJ raiz                   Só PJ   Posterior em até 12 meses  4/91     4.4  252/6672         3.8      3.4      1.174                 0.78                 0.74
Fornecedor     CNPJ raiz        


=== Concentração por fornecedor (CNPJ raiz, pessoa jurídica) ===
Fornecedores PJ distintos: 6,672 | propostas por fornecedor: mediana 2, máximo 708
Fornecedores com sanção no CEIS: 770 | propostas dessas: 11,764 | maior fornecedor: 708 propostas | os 20 maiores respondem por 33% das propostas com correspondência
Propostas por fornecedor: sancionados média 15.3 (mediana 4) × demais média 9.6 (mediana 2)
Alto risco: 306 propostas de 96 fornecedores distintos | o maior fornecedor tem 99 propostas (32%) | os 5 maiores, 53%



Referência ponderada por proposta: sorteando 306 propostas ao acaso, 16.8% dos fornecedores PJ distintos sorteados são sancionados (IC 95% 12.7% a 21.0%); na população de fornecedores PJ, 11.5%.


---
### 5. Discussão

**Leitura (rodada atual).** Resultados por proposta e por fornecedor são apresentados lado a lado porque a unidade de análise muda a força da evidência.

#### 5.1 Sanção vigente na data da proposta × sanção posterior (seção 3.2)

- **O "15% de propostas com fornecedor no CEIS" é, quase todo, sanção posterior à proposta.** Das 11.764 propostas cujo fornecedor (raiz do CNPJ) consta no CEIS, só **154 (1,3%)** tinham sanção **vigente na data**; 11.602 (98,6%) têm apenas sanção **posterior**. Isso acontece porque 9.926 das 14.533 sanções de pessoas jurídicas (68,3%) começaram depois da última proposta (30/12/2024), com mediana de **18,6 meses** depois da proposta.
  A classificação anterior (ativa em 05/10/2026) reunia 11.308 propostas; destas, 11.166 (98,7%) só tinham sanção posterior (CNPJ completo). Ela media, na prática, "fornecedor sancionado depois".
- **Sanção vigente na data (sinal forte): nenhuma das 306 propostas de alto risco.** Esperavam-se 0,6 (154 de 75.471 propostas, de 24 fornecedores), então **o teste não tem poder**: 0 observado é compatível com o acaso (p unilateral = 0,53) e não é evidência nem a favor nem contra.
  Das 154 propostas vigentes, 99 são impedimentos de contratar, 49 suspensões e 6 declarações de inidoneidade; 28 valem apenas "no órgão sancionador" e 97 vêm de órgãos municipais. O sinal só é forte se a sanção alcançar o órgão que contratou, o que os dados não permitem verificar.
- **Sanção posterior (detecção antecipada): o alto risco foi menos sancionado depois.** 6 de 306 propostas (2,0%) contra 15,5% na base (odds ratio 0,11; p < 10⁻¹⁴). Na janela fixa de 12 meses, com a mesma exposição para todas as propostas, 2 de 306 (0,7%) contra 3,6% (odds ratio 0,18; p = 0,0018). Considerando só pessoas jurídicas, 3,6% contra 17,1% e 1,2% contra 4,0% (p bilateral 0,071 na janela de 12 meses).
- **CNPJ raiz × completo:** a raiz acrescenta 90 propostas (10 vigentes, 80 posteriores, 48 na janela de 12 meses) e 28 raízes têm mais de um CNPJ sancionado. O número de propostas de alto risco com correspondência continua 6, de modo que a conclusão não depende da chave.

#### 5.2 Por proposta × por fornecedor (seção 4)

| Comparação (CNPJ raiz) | Alto risco | Base | Esperado | Odds ratio | p bilateral | p unilateral (menos) |
|---|---|---|---|---|---|---|
| **Proposta**, todas | 6/306 (2,0%) | 15,6% | 47,7 | 0,11 | 2,8×10⁻¹⁵ | 1,3×10⁻¹⁵ |
| **Fornecedor**, só PJ | 5/91 (5,5%) | 11,5% | 10,5 | 0,44 | 0,070 | 0,040 |
| **Fornecedor**, todos (PJ + PF) | 5/96 (5,2%) | 8,2% | 7,9 | 0,61 | 0,35 | 0,19 |
| **Fornecedor**, só PJ, posterior em até 12 meses | 4/91 (4,4%) | 3,8% | 3,4 | 1,17 | 0,78 | 0,74 |

- **A unidade de análise muda a força da evidência.** Por proposta, a associação inversa é extrema (p ≈ 10⁻¹⁵); por fornecedor, tem o mesmo sentido, mas é marginal (p bilateral 0,07, unilateral 0,04). A diferença vem da concentração: o alto risco tem 306 propostas de 96 fornecedores, e um deles responde por 99 (32%); os 5 maiores, por 53%. Do lado da base, o maior fornecedor tem 708 propostas e os 20 maiores respondem por 33% das propostas com correspondência.
- **O resultado por fornecedor depende da população.** Incluir as pessoas físicas, que nunca casam por construção (o CEIS foi filtrado por pessoa jurídica), reduz a taxa da população de 11,5% para 8,2% e o p sobe para 0,35: não se distingue do acaso. A versão só com pessoas jurídicas é a que compara fornecedores que *poderiam* casar.
- **Na janela de 12 meses, por fornecedor, a associação inversa desaparece** (4 de 91 contra 3,4 esperados; p = 0,78).
- **Baixo poder.** Com 5 fornecedores sancionados no alto risco, nenhuma conclusão por fornecedor é firme. O mesmo vale para a sanção vigente (0 de 91; esperado 0,3).
- **O teste por fornecedor é favorável ao alto risco.** Os fornecedores sancionados são maiores (média de 15,3 propostas contra 9,6; mediana 4 contra 2), e o alto risco é selecionado por proposta. Sorteando 306 propostas ao acaso, 16,8% dos fornecedores PJ sorteados seriam sancionados (IC 95% 12,7% a 21,0%), contra 5,5% observados; a população de fornecedores (11,5%) é a referência mais favorável.

#### 5.3 O que se pode e não se pode afirmar

- **Há associação inversa** entre alto risco e sanção posterior, mais forte por proposta do que por fornecedor; por proposta, a significância é inflada pela concentração em poucos fornecedores.
- **Não há evidência de detecção antecipada**, e a evidência de que o alto risco concentre empresas já impedidas na data da proposta é inconclusiva (o teste não tem poder).
- Esse resultado não invalida os modelos: o CEIS mede sanção formal, e anomalia estatística não é irregularidade.

| Hipótese | Explicação | Situação nesta análise |
|---|---|---|
| **Perfil "normal" dos sancionados** | Fornecedores grandes e recorrentes concentram as sanções e têm perfil típico, não anômalo. | Compatível: sancionados têm mais propostas por fornecedor (15,3 contra 9,6). *Inferência.* |
| **Detecção prospectiva** | Os modelos identificariam empresas suspeitas ainda não sancionadas. | Não sustentada: o alto risco foi menos sancionado depois, por proposta (e sem diferença por fornecedor na janela de 12 meses). |
| **Perfil diferente de anomalia** | Preço, competição e calendário capturam padrões distintos dos que levam a sanções formais, que frequentemente decorrem de processo administrativo e da execução do contrato. | Não testado. A sanção posterior pode ser consequência do próprio contrato (inexecução); sem dados de execução, isso não pode ser verificado. *Inferência.* |
| **Limitação do CEIS** | Só uma fração das irregularidades chega a sanção publicada, e sanções de órgão municipal podem não alcançar o órgão que contratou. | 28 das 154 sanções vigentes na data valem só no órgão sancionador; as demais têm abrangência maior. |
